# LISA Traffic Light Detection with YOLOv8
## Complete Workflow: From Dataset Exploration to Model Export

This notebook walks you through the entire process of building a traffic light detector using the LISA dataset and YOLOv8:
1. **Explore** the raw LISA dataset structure and annotations
2. **Convert** Supervisly JSON annotations to YOLO format
3. **Prepare** the dataset with train/val/test splits
4. **Train** a YOLOv8 object detection model
5. **Evaluate** and run inference on the model
6. **Export** the model to ONNX format for deployment

**Important: Run cells in order!** The dataset is pre-split into `Dataset/train/` and `Dataset/test/`. We'll create a validation split from train.

In [ ]:
# 1. SETUP AND IMPORTS
import os
import sys
import json
import shutil
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# Project paths
PROJECT_ROOT = Path.cwd()
DATASET_DIR = PROJECT_ROOT / "Dataset"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RUNS_DIR = PROJECT_ROOT / "runs"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Dataset Dir: {DATASET_DIR}")
print(f"Processed Data Dir: {DATA_PROCESSED_DIR}")

# Create output directories if they don't exist
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Directories ready")

## Step 1: Explore the LISA Dataset Structure

Let's first understand what we're working with:
- Dataset is organized in `Dataset/train/` and `Dataset/test/`
- Each split has `img/` (images) and `ann/` (Supervisly JSON annotations)
- Annotations contain bounding boxes with class labels

In [ ]:
# 2. EXPLORE DATASET STRUCTURE
print("="*60)
print("DATASET STRUCTURE")
print("="*60)

# Check meta.json
with open(DATASET_DIR / "meta.json") as f:
    meta = json.load(f)

print(f"\nClasses in Dataset ({len(meta['classes'])} total):")
CLASS_NAMES = []
for idx, cls in enumerate(meta['classes']):
    CLASS_NAMES.append(cls['title'])
    print(f"  {idx}: {cls['title']}")

CLASS_MAP = {name: idx for idx, name in enumerate(CLASS_NAMES)}
print(f"\n✓ Total classes: {len(CLASS_NAMES)}")

# Explore train split
train_img_dir = DATASET_DIR / "train" / "img"
train_ann_dir = DATASET_DIR / "train" / "ann"
train_images = sorted([f for f in os.listdir(train_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
train_anns = sorted([f for f in os.listdir(train_ann_dir) if f.endswith('.json')])

print(f"\nTrain Split:")
print(f"  Images: {len(train_images)}")
print(f"  Annotations: {len(train_anns)}")
if train_images:
    print(f"  Sample images: {train_images[:3]}")

# Explore test split
test_img_dir = DATASET_DIR / "test" / "img"
test_ann_dir = DATASET_DIR / "test" / "ann"
test_images = sorted([f for f in os.listdir(test_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
test_anns = sorted([f for f in os.listdir(test_ann_dir) if f.endswith('.json')])

print(f"\nTest Split:")
print(f"  Images: {len(test_images)}")
print(f"  Annotations: {len(test_anns)}")
if test_images:
    print(f"  Sample images: {test_images[:3]}")

print(f"\nTotal images across all splits: {len(train_images) + len(test_images)}")

## Step 2: Understand the Annotation Format

Let's examine annotation files to understand the Supervisly JSON structure.

In [ ]:
# 3. EXAMINE ANNOTATION FORMAT
print("="*60)
print("ANNOTATION FORMAT EXAMPLE")
print("="*60)

# Load one annotation to understand structure
sample_ann_file = train_ann_dir / train_anns[0]
with open(sample_ann_file) as f:
    sample_ann = json.load(f)

print(f"\nAnnotation file: {train_anns[0]}")
print(f"Image dimensions: {sample_ann['size']['width']} x {sample_ann['size']['height']}")
print(f"Number of objects: {len(sample_ann.get('objects', []))}")

print("\nFirst object structure:")
if sample_ann.get('objects'):
    obj = sample_ann['objects'][0]
    print(f"  classTitle: {obj['classTitle']}")
    print(f"  geometryType: {obj['geometryType']}")
    print(f"  points.exterior: {obj['points']['exterior']}")
    print(f"  Format: [[x1,y1], [x2,y2]] (top-left, bottom-right)")

# Show all classes in this annotation
class_set = set()
for obj in sample_ann.get('objects', []):
    class_set.add(obj['classTitle'])

print("\nClasses in this image:", class_set)

## Step 3: Visualize Annotations

Let's visualize sample images with their bounding boxes.

In [ ]:
# 4. VISUALIZE ANNOTATION
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (split_name, img_dir, ann_dir, anns) in enumerate([
    ("Train", train_img_dir, train_ann_dir, train_anns),
    ("Test", test_img_dir, test_ann_dir, test_anns)
]):
    if not anns or len(anns) < 6:
        continue
    
    # Load image and annotation
    ann_file = ann_dir / anns[5]  # Pick 5th image for variety
    img_name = str(ann_file.name).replace('.json', '')
    img_path = img_dir / img_name
    
    if not img_path.exists():
        continue
    
    img = Image.open(img_path)
    with open(ann_file) as f:
        ann = json.load(f)
    
    ax = axes[idx]
    ax.imshow(img)
    ax.set_title(f"{split_name} Split - {img_name}")
    ax.axis('off')
    
    # Draw bounding boxes
    colors = plt.cm.get_cmap('tab20')(np.linspace(0, 1, len(CLASS_NAMES)))
    color_map = {name: colors[i] for i, name in enumerate(CLASS_NAMES)}
    
    for obj in ann.get('objects', []):
        pts = obj['points']['exterior']
        x1, y1 = pts[0]
        x2, y2 = pts[1]
        x_min = min(x1, x2)
        y_min = min(y1, y2)
        width = abs(x2 - x1)
        height = abs(y2 - y1)
        
        class_title = obj['classTitle']
        color = color_map.get(class_title, (1, 1, 1))
        
        rect = Rectangle((x_min, y_min), width, height, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x_min, y_min - 5, class_title, fontsize=8, color=color, weight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualized sample annotations")

## Step 4: Convert LISA Annotations to YOLO Format

YOLO format requires:
- For each image: a `.txt` file with one line per object
- Format: `class_id x_center y_center width height`
- All coordinates normalized to [0, 1]

In [ ]:
# 5. CONVERT LISA ANNOTATIONS TO YOLO FORMAT
print("="*60)
print("CONVERTING ANNOTATIONS TO YOLO FORMAT")
print("="*60)

def convert_annotation_to_yolo(ann_data, class_map):
    """Convert Supervisly annotation to YOLO format"""
    img_h = ann_data["size"]["height"]
    img_w = ann_data["size"]["width"]
    
    yolo_lines = []
    for obj in ann_data.get("objects", []):
        class_title = obj.get("classTitle", "").strip()
        if not class_title or class_title not in class_map:
            continue
        
        pts = obj["points"]["exterior"]
        if len(pts) != 2:
            continue
        
        x1, y1 = pts[0]
        x2, y2 = pts[1]
        
        x_min = min(x1, x2)
        x_max = max(x1, x2)
        y_min = min(y1, y2)
        y_max = max(y1, y2)
        
        # Normalize to [0, 1]
        x_center = ((x_min + x_max) / 2.0) / img_w
        y_center = ((y_min + y_max) / 2.0) / img_h
        width = (x_max - x_min) / img_w
        height = (y_max - y_min) / img_h
        
        # Clamp to [0, 1]
        x_center = max(0, min(1, x_center))
        y_center = max(0, min(1, y_center))
        width = max(0, min(1, width))
        height = max(0, min(1, height))
        
        class_id = class_map[class_title]
        yolo_line = f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"
        yolo_lines.append(yolo_line)
    
    return yolo_lines

def process_split(split_name, img_dir, ann_dir, out_img_dir, out_lbl_dir, class_map, use_symlink=True):
    """Process one split (train/test) and convert to YOLO format"""
    print(f"\nProcessing {split_name} split...")
    
    ann_files = sorted([f for f in os.listdir(ann_dir) if f.endswith(".json")])
    processed = 0
    
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)
    
    for ann_file in ann_files:
        ann_path = os.path.join(ann_dir, ann_file)
        img_name = ann_file.replace(".json", "")
        img_path = os.path.join(img_dir, img_name)
        
        if not os.path.exists(img_path):
            continue
        
        # Convert annotation
        with open(ann_path) as f:
            ann_data = json.load(f)
        yolo_lines = convert_annotation_to_yolo(ann_data, class_map)
        
        # Create symlink or copy for image
        out_img_path = os.path.join(out_img_dir, img_name)
        if not os.path.exists(out_img_path):
            if use_symlink:
                os.symlink(os.path.abspath(img_path), out_img_path)
            else:
                shutil.copy2(img_path, out_img_path)
        
        # Write YOLO label file
        lbl_name = img_name + ".txt"
        out_lbl_path = os.path.join(out_lbl_dir, lbl_name)
        with open(out_lbl_path, "w") as f:
            f.write("\n".join(yolo_lines))
            if yolo_lines:
                f.write("\n")
        
        processed += 1
    
    print(f"  ✓ Processed {processed} images from {split_name}")
    return processed

# Process train split
train_processed = process_split(
    "Train",
    img_dir=str(train_img_dir),
    ann_dir=str(train_ann_dir),
    out_img_dir=str(DATA_PROCESSED_DIR / "images" / "train"),
    out_lbl_dir=str(DATA_PROCESSED_DIR / "labels" / "train"),
    class_map=CLASS_MAP,
    use_symlink=True
)

# Process test split
test_processed = process_split(
    "Test",
    img_dir=str(test_img_dir),
    ann_dir=str(test_ann_dir),
    out_img_dir=str(DATA_PROCESSED_DIR / "images" / "test"),
    out_lbl_dir=str(DATA_PROCESSED_DIR / "labels" / "test"),
    class_map=CLASS_MAP,
    use_symlink=True
)

print(f"\n✓ Total: {train_processed + test_processed} images converted")

## Step 5: Create Validation Split

We'll sample 10% of training images for validation during model tuning.

In [ ]:
# 6. CREATE VALIDATION SPLIT (10% from train)
print("="*60)
print("CREATING VALIDATION SPLIT")
print("="*60)

train_img_processed = DATA_PROCESSED_DIR / "images" / "train"
train_lbl_processed = DATA_PROCESSED_DIR / "labels" / "train"
val_img_dir = DATA_PROCESSED_DIR / "images" / "val"
val_lbl_dir = DATA_PROCESSED_DIR / "labels" / "val"

os.makedirs(val_img_dir, exist_ok=True)
os.makedirs(val_lbl_dir, exist_ok=True)

# Get all train images
train_images_list = sorted([f for f in os.listdir(train_img_processed) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.jpg'))])
n_val = max(1, int(len(train_images_list) * 0.1))
val_images = set(random.sample(train_images_list, n_val))

print(f"\nTotal train images: {len(train_images_list)}")
print(f"Moving to val: {len(val_images)} ({100*len(val_images)/len(train_images_list):.1f}%)")

# Move images and labels to val
moved = 0
for img_name in val_images:
    # Move image
    src_img = train_img_processed / img_name
    dst_img = val_img_dir / img_name
    if src_img.exists():
        if os.path.islink(src_img):
            target = os.readlink(src_img)
            os.symlink(target, dst_img)
            os.remove(src_img)
        else:
            shutil.move(str(src_img), str(dst_img))
    
    # Move label
    lbl_name = img_name + ".txt"
    src_lbl = train_lbl_processed / lbl_name
    dst_lbl = val_lbl_dir / lbl_name
    if src_lbl.exists():
        shutil.move(str(src_lbl), str(dst_lbl))
    
    moved += 1

# Verify splits
final_train = len([f for f in os.listdir(train_img_processed) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
final_val = len([f for f in os.listdir(val_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
final_test = len([f for f in os.listdir(DATA_PROCESSED_DIR / "images" / "test") if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f"\nFinal split:")
print(f"  Train: {final_train} images")
print(f"  Val:   {final_val} images")
print(f"  Test:  {final_test} images")
print(f"  Total: {final_train + final_val + final_test} images")

## Step 6: Create YOLOv8 Configuration (data.yaml)

In [ ]:
# 7. CREATE DATA.YAML FOR YOLOV8
print("="*60)
print("CREATING YOLOv8 CONFIGURATION")
print("="*60)

yaml_content = f"""path: data/processed
train: images/train
val: images/val
test: images/test

nc: {len(CLASS_NAMES)}
names:
""" + "\n".join([f"  {idx}: {name}" for idx, name in enumerate(CLASS_NAMES)])

yaml_path = DATA_PROCESSED_DIR / "data.yaml"
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"\n✓ Created: {yaml_path}")
print(f"Classes: {len(CLASS_NAMES)}")
print("\nContent preview:")
print(yaml_content[:300])

## Step 7: Install YOLOv8

In [ ]:
# 8. INSTALL YOLOV8
import subprocess
import sys

print("="*60)
print("INSTALLING YOLOV8")
print("="*60)

try:
    import ultralytics
    print("✓ ultralytics already installed")
except ImportError:
    print("Installing ultralytics...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    print("✓ ultralytics installed")

print("\n✓ Ready to train")

## Step 8: Train YOLOv8 Model

Training will start the object detection model learning on the prepared dataset. For quick testing, we'll train for just 10 epochs. For production, increase this to 50-100.

In [ ]:
# 9. TRAIN YOLOV8 MODEL
from ultralytics import YOLO

print("="*60)
print("TRAINING YOLOV8 MODEL")
print("="*60)

# Create model (nano for quick testing)
model = YOLO('yolov8n.pt')

print("\nStarting training...")
print("(This may take several minutes)\n")

# Train
results = model.train(
    data=str(DATA_PROCESSED_DIR / 'data.yaml'),
    epochs=10,
    imgsz=640,
    batch=16,
    patience=5,
    device=0,
    project=str(RUNS_DIR),
    name='traffic_light_v1',
    verbose=False
)

print("\n✓ Training complete!")
print(f"Results saved to: {RUNS_DIR}")

## Step 9: Run Inference on Sample Images

In [ ]:
# 10. RUN INFERENCE
print("="*60)
print("RUNNING INFERENCE ON TEST IMAGES")
print("="*60)

# Load best model
best_model_paths = list(RUNS_DIR.glob("**/weights/best.pt"))
if best_model_paths:
    best_model_path = best_model_paths[0]
    model = YOLO(str(best_model_path))
    
    test_img_dir = DATA_PROCESSED_DIR / "images" / "test"
    test_images = sorted([f for f in os.listdir(test_img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:3]
    
    if test_images:
        print(f"\nRunning inference on {len(test_images)} test images...")
        
        results = model.predict(
            source=[str(test_img_dir / img) for img in test_images],
            save=True,
            conf=0.5,
            project=str(RUNS_DIR),
            name='test_predictions'
        )
        
        print(f"✓ Done! Predictions saved")
        print(f"Check {RUNS_DIR}/test_predictions for annotated images")
else:
    print("No trained model found. Complete training first.")

## Step 10: Export Model to ONNX

Export the trained model for deployment and use in other frameworks.

In [ ]:
# 11. EXPORT MODEL
print("="*60)
print("EXPORTING MODEL TO ONNX")
print("="*60)

best_model_paths = list(RUNS_DIR.glob("**/weights/best.pt"))
if best_model_paths:
    best_model_path = best_model_paths[0]
    model = YOLO(str(best_model_path))
    
    print("\nExporting to ONNX format...")
    model.export(format='onnx', imgsz=640)
    
    # Copy best model to models directory
    pt_path = MODELS_DIR / "traffic_light_detector.pt"
    shutil.copy2(str(best_model_path), str(pt_path))
    print(f"✓ Saved PyTorch model to: {pt_path}")
    
    print("\n✓ Export complete!")
    print(f"Training artifacts in: {RUNS_DIR}")
    print(f"Final model in: {MODELS_DIR}")
else:
    print("No trained model found.")

## Summary

✅ **Complete Workflow Summary**

You've successfully:
1. Explored the LISA Traffic Light Dataset (14 classes, pre-split train/test)
2. Understood Supervisly JSON annotation format
3. Converted annotations to YOLO normalized format
4. Created train/val/test splits with proper organization
5. Generated YOLOv8 configuration file
6. Trained YOLOv8 object detection model
7. Ran inference and verified predictions
8. Exported model to ONNX format

### Output Locations
- **Dataset**: `data/processed/images/` and `data/processed/labels/`
- **Configuration**: `data/processed/data.yaml`
- **Training Logs**: `runs/detect/traffic_light_v1/`
- **Models**: `models/traffic_light_detector.pt` and `.onnx`

### Next Steps
- Train longer (50-100 epochs) for better accuracy
- Use larger models (yolov8s/m/l) for production
- Deploy using ONNX model for inference
- Fine-tune on your specific use case